# 2 · The naive baseline · `full_realtime`

The simplest possible bid path:

1. resolve identity,
2. fetch the full MAID profile,
3. **fetch every active campaign**,
4. filter all of them in app memory,
5. rerank the survivors.

Nothing wrong with the *correctness* of this path — it's the reference
the other modes are checked against. Everything wrong with the *latency*.


In [1]:
# Locate the repo root so `notebooks._demo_setup` is importable regardless
# of where the kernel was launched (the package layout requires the repo
# root on sys.path).
import sys
from pathlib import Path
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from notebooks._demo_setup import connect_redis, StepTimer
client = connect_redis()

connected to redis://localhost:6381/0
  users=4000  campaigns=2500  precompute_version=v17_2500_12


## Step-by-step

The bid engine walks the request through the steps in order. The timer
captures the wall-clock cost of each.


In [2]:
IDENTITY_TOKEN = 'id_00042_01'
timer = StepTimer()

with timer.step('identity_resolution'):
    maid_id = client.get(f'identity:{IDENTITY_TOKEN}')

with timer.step('profile_fetch'):
    profile = client.hgetall(f'maid:{maid_id}')

with timer.step('campaign_id_scan'):
    # full_realtime materializes the entire campaign universe.
    campaign_ids = sorted(
        key.removeprefix('campaign:')
        for key in client.scan_iter(match='campaign:*', count=2000)
    )

with timer.step('campaign_fetch_pipelined'):
    pipe = client.pipeline(transaction=False)
    for cid in campaign_ids:
        pipe.hgetall(f'campaign:{cid}')
    payloads = pipe.execute()

print(f'maid_id        = {maid_id}')
print(f'profile fields = {len(profile)}')
print(f'campaigns seen = {len(campaign_ids)}')
print()
print(timer.summary())

maid_id        = maid_00042
profile fields = 14
campaigns seen = 2500

             identity_resolution    0.370 ms
                   profile_fetch    0.237 ms
                campaign_id_scan   40.557 ms
        campaign_fetch_pipelined   76.821 ms
--------------------------------------------
                           TOTAL  117.985 ms


## Filter + rerank in app memory

The prototype's `filter_campaigns_for_user` function holds the same
eligibility logic for every mode — geo, state, device, card tier, segment
required/any_of/none_of, pacing, budget, frequency, **and** the float-score
`taxonomy_filter`. Running it over all 2500 campaigns is the work that
`full_realtime` is paying for.


In [3]:
from app.candidate import filter_campaigns_for_user
from app.models import Campaign, UserProfile
from app.ranking import rerank_campaigns

# Re-hydrate the profile + campaigns into prototype model instances.
user = UserProfile.from_redis_hash(profile)
campaigns = [Campaign.from_redis_hash(p) for p in payloads if p]

with timer.step('filter_in_app'):
    eligible = filter_campaigns_for_user(user, campaigns)

with timer.step('rerank'):
    top_5 = rerank_campaigns(user, eligible, top_k=5)

print(f'filtered  {len(campaigns)} -> {len(eligible)} eligible')
print('top 5 ranked:')
for ranked in top_5:
    print(f'  {ranked.campaign_id}  score={ranked.score:.4f}')
print()
print(timer.summary())

filtered  2500 -> 10 eligible
top 5 ranked:
  c00848  score=4.8107
  c01551  score=4.4065
  c01222  score=4.2812
  c02229  score=4.1223
  c01617  score=3.6611

             identity_resolution    0.370 ms
                   profile_fetch    0.237 ms
                campaign_id_scan   40.557 ms
        campaign_fetch_pipelined   76.821 ms
                   filter_in_app    1.299 ms
                          rerank    0.070 ms
--------------------------------------------
                           TOTAL  119.354 ms


## What the timing tells us

A few things to point out on the call:

- **`campaign_fetch_pipelined`** dominates. Pulling 2500 hashes back is one
  pipelined round trip but a few thousand actual Redis ops, and the response
  payload is non-trivial.
- **`filter_in_app`** is also significant — every one of those 2500
  campaigns gets the full eligibility check.
- The decision-path total here is the number every other mode is trying
  to beat. The published headline runs the same path on a tuned VM and
  comes in around `~40 ms` p50. The remaining notebooks show how the
  prototype gets that down to single-digit milliseconds.

Note: this is `concurrency = 1` running on whatever shape the local docker
stack happens to be on. The relative ordering of modes is what matters
in this demo, not the absolute numbers — see `reports/benchmark_report.md`
for the tuned VM run.
